# Catch the Shadowban 🕵️
### A hands-on audit of social media visibility

Social media platforms decide what you see, but their recommendation algorithms are black boxes.
When a platform silently reduces the reach of some content (*shadowbanning*), how can an outsider prove it?

In this notebook you will:
1. see **why visibility data on social networks look the way they do** (heavy tails, log-normals);
2. compute the **p-score**, a visibility metric, on **real Twitter/X data**;
3. audit a fictional platform, **Chirper**, where four groups of users claim to be shadowbanned.
   Only **one** of them is right.

Cells marked `# TODO` are yours to complete. Replace the `...` with your code.

*Based on: A. Galeazzi, P. Paudel, M. Conti, E. De Cristofaro, G. Stringhini,
"Revealing The Secret Power: How Algorithms Can Influence Content Visibility on Twitter/X", NDSS 2026.*

In [ ]:
# Setup: run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

DATA_URL = "https://raw.githubusercontent.com/YOUR_USER/YOUR_REPO/main/data/"   # set before the lecture

def load(name):
    """Load a dataset from DATA_URL (or from a file uploaded to Colab)."""
    try:
        return pd.read_csv(DATA_URL + name)
    except Exception:
        return pd.read_csv(name)

def log_hist(values, groups=None, labels=None, bins=50, ax=None, xlabel=""):
    """Histogram with logarithmic bins: the fraction of items per bin, on a log x-axis."""
    values = np.asarray(values, dtype=float)
    pos = values[values > 0]
    edges = np.logspace(np.log10(pos.min()), np.log10(pos.max()), bins)
    if np.all(pos == np.round(pos)):   # integer counts: avoid empty bins between 1, 2, 3...
        edges = np.unique(np.concatenate([[pos.min() - 0.5], np.round(edges) + 0.5]))
    ax = ax or plt.gca()
    if groups is None:
        ax.hist(pos, bins=edges, weights=np.ones(len(pos)) / len(pos), alpha=0.8)
    else:
        groups = np.asarray(groups)
        for g in np.unique(groups):
            v = values[(groups == g) & (values > 0)]
            name = labels[g] if labels else str(g)
            ax.hist(v, bins=edges, weights=np.ones(len(v)) / len(v), alpha=0.5, label=name)
        ax.legend()
    ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("fraction")
    return ax

---
## Part 1 · Networks in ten minutes

### 1.1 Who sees your posts?
A social platform is a **directed network**: an edge *u → v* means that *u* follows *v*.
The number of followers of an account is its **in-degree**, and it is the first thing that determines
how many people *can* see a post.

Follower networks grow by **preferential attachment**: new users tend to follow accounts that are
already popular ("the rich get richer"). The result is a **heavy-tailed** degree distribution:
most accounts have a handful of followers, a few have millions.

Let's simulate one with the Barabási–Albert model.

In [ ]:
G = nx.barabasi_albert_graph(n=20000, m=2, seed=1)
degrees = np.array([d for _, d in G.degree()])
print(f"median degree: {np.median(degrees):.0f}   mean: {degrees.mean():.1f}   max: {degrees.max()}")

plt.hist(degrees, bins=100)
plt.xlabel("degree (followers)"); plt.ylabel("number of accounts")
plt.title("Linear axes");

On linear axes we see one tall bar and nothing else: the interesting structure is squeezed against zero.

In [ ]:
# TODO: plot the same degree distribution with log_hist (log bins, log x-axis),
#       then put the y-axis on a log scale too (plt.yscale("log")).
...

### 1.2 Visibility is a multiplicative process
Followers are only the start. A post's views are roughly a **product** of factors:

$$\text{views} \approx \text{followers} \times P(\text{shown in the feed}) \times P(\text{not scrolled past}) \times \text{amplification by shares} \times \dots$$

Take the logarithm and the product becomes a **sum**:
$\log(\text{views}) = \log(\text{followers}) + \log P(\dots) + \dots$

By the central limit theorem, a sum of many independent terms is approximately normal.
So $\log(\text{views})$ is approximately normal, i.e. views are **log-normal**.
Let's check it with a toy simulation.

In [ ]:
rng = np.random.default_rng(0)
n_posts, n_factors = 50_000, 12

# TODO: build `views` as the product of n_factors independent random factors per post,
#       e.g. uniform between 0.2 and 3 (hint: rng.uniform(..., size=(n_posts, n_factors)).prod(axis=1))
views_sim = ...

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(views_sim, bins=100); axes[0].set_title("Linear scale")
log_hist(views_sim, ax=axes[1]); axes[1].set_title("Log scale")

### 1.3 What this means for an auditor
* A shadowban is **multiplicative**: "this post gets 3× fewer views". On a log scale it is a **shift**
  of the whole distribution by $\log 3$.
* **Means are dominated by a few viral posts.** Use **medians** and **ratios of medians**.
* Distributions are far from normal, so use **non-parametric tests** (Mann–Whitney U) or work on $\log$ values.

---
## Part 2 · Real data: Twitter/X and links

A sample of original tweets about military aid to Ukraine (Nov 2022 – Mar 2023), with view counts.
Data are anonymized: random author labels, no text, no tweet IDs, rounded counts.

| column | meaning |
|---|---|
| `author` | anonymous author label |
| `followers` | author's followers |
| `views` | impressions of the tweet |
| `likes`, `retweets` | engagement |
| `has_link` | 1 if the tweet links to a website outside Twitter/X |

In [ ]:
tw = load("twitter_sample.csv.gz")
print(len(tw), "tweets by", tw.author.nunique(), "authors")
tw.head()

In [ ]:
# TODO: plot a histogram of tw.views with plt.hist on LINEAR axes. What can you see?
...

In [ ]:
# TODO: now the same with log_hist. What shape do you recognise?
...

### 2.1 The p-score
Views depend heavily on how many followers the author has. To compare tweets by different authors we normalise:

$$p\text{-score} = \frac{\text{views}}{\text{followers}}$$

A high p-score means more views per follower; a low one means limited circulation.

In [ ]:
# TODO: add a column "pscore" to tw and plot its distribution for tweets with and without links
tw["pscore"] = ...

log_hist(tw.pscore, groups=tw.has_link, labels={0: "no link", 1: "link"}, xlabel="p-score");

In [ ]:
# TODO: compare the two groups
#   1. mean and median p-score for has_link = 0 and has_link = 1
#   2. the ratio of the medians (no link / link)
#   3. a one-sided Mann–Whitney U test: are no-link p-scores larger?
...

**Case closed? Twitter penalises links?** Not so fast.
The p-score tells us that link tweets get fewer views *per follower*. It does **not** tell us *why*.
Let's train our eye on a platform where we know the truth.

---
## Part 3 · Catch the shadowban on Chirper

Chirper is a (fictional) social network. Four groups of users have filed complaints:

| case | complaint |
|---|---|
| **A** | "Posts tagged **#news** are shadowbanned!" |
| **B** | "Chirper buries **memes**!" |
| **C** | "Nobody sees my **long-form** posts anymore!" |
| **D** | "**Giveaway** posts are being suppressed!" |

You get the data for each case. `in_category = 1` marks the posts in the category under suspicion.
**Exactly one** complaint is a real platform penalty. Find it.

In [ ]:
cases = {k: load(f"case_{k}.csv.gz") for k in "ABCD"}
cases["A"].head()

### 3.1 The naive audit

In [ ]:
def naive_ratio(df):
    """Median p-score of the other posts divided by the median p-score of the category.
    > 1 means the category gets fewer views per follower."""
    # TODO
    ...

for k, df in cases.items():
    print(k, round(naive_ratio(df), 2))

All four categories get roughly **3× fewer views per follower**. By the p-score, all four look guilty.
Three of them are innocent. We need better forensic tools.

### 3.2 Check 1: popularity
The p-score assumes that views are *proportional* to followers. Is that true? Fit
$\log(\text{views}) = c + \alpha \log(\text{followers})$: if $\alpha < 1$, big accounts get
fewer views *per follower* even without any penalty.

In [ ]:
def alpha(df):
    """Slope of log(views) vs log(followers)."""
    # TODO (hint: np.polyfit(x, y, 1) returns [slope, intercept])
    ...

def popularity_check(df):
    """Penalty factor for the category after controlling for followers.
    Regress log(views) on log(followers) and in_category; return exp(-coefficient)."""
    # TODO (hint: smf.ols("np.log(views) ~ np.log(followers) + in_category", data=df).fit().params)
    ...

for k, df in cases.items():
    print(k, f"alpha = {alpha(df):.2f}   penalty controlling for followers = {popularity_check(df):.2f}x")

### 3.3 Check 2: engagement
Maybe people simply **like the category less**. Recommendation algorithms amplify engaging content,
so less engaging content gets fewer views even with no targeted penalty.
A pure penalty reduces *how many* people see a post, not how much *those who see it* like it:
compare the **like rate** (likes per view).

In [ ]:
def engagement_check(df, min_views=100):
    """Median like rate (likes / views) of the other posts divided by that of the category.
    Use only posts with at least `min_views` views: on tiny posts the like rate is pure noise."""
    # TODO
    ...

for k, df in cases.items():
    print(k, f"like-rate ratio = {engagement_check(df):.2f}")

### 3.4 Check 3: who is posting?
A category can look penalised because of **one account**: if a single low-visibility account produces
most of the posts, it drags the median down.

In [ ]:
def concentration_check(df):
    """Return (share of category posts written by its most active author,
               naive_ratio after removing that author)."""
    # TODO
    ...

for k, df in cases.items():
    share, ratio = concentration_check(df)
    print(k, f"top author share = {share:.2f}   naive ratio without them = {ratio:.2f}")

### 3.5 The verdict

In [ ]:
summary = pd.DataFrame({
    k: {"naive": naive_ratio(df),
        "ctrl. followers": popularity_check(df),
        "like-rate ratio": engagement_check(df),
        "top author share": concentration_check(df)[0],
        "naive w/o top author": concentration_check(df)[1]}
    for k, df in cases.items()}).T.round(2)
summary

**Your verdict.** For each case: penalised or not? If not, what explains the gap?

| case | verdict | explanation |
|---|---|---|
| A | | |
| B | | |
| C | | |
| D | | |

---
## Part 4 · Back to Twitter

Apply the same checks to the real data, with `has_link` as the category under suspicion.
Does the link penalty survive?

In [ ]:
tw_case = tw.rename(columns={"has_link": "in_category"})
print(f"naive ratio            : {naive_ratio(tw_case):.2f}x")
print(f"alpha                  : {alpha(tw_case):.2f}")
print(f"ctrl. for followers    : {popularity_check(tw_case):.2f}x")
print(f"like-rate ratio        : {engagement_check(tw_case):.2f}")
share, ratio = concentration_check(tw_case)
print(f"top author share       : {share:.2f}   naive ratio without them: {ratio:.2f}x")

---
## Part 5 · From checks to a model

Our checks worked on Chirper because each trap had **one** cause. Real platforms mix all of them:
popularity, account quality, engagement and penalties act **at the same time**, and checking them one at a time
does not tell us how much of the visibility gap to attribute to each.

A first step is to compare posts **by the same author**: this cancels out followers, fake followers
and everything else that is constant for an account.

In [ ]:
def within_author_ratio(df):
    """Median, across authors who post in both groups, of (median p-score other / median p-score category)."""
    lp = np.log(df.views / df.followers)
    g = df.assign(lp=lp).groupby(["author", "in_category"]).lp.median().unstack().dropna()
    return np.exp((g[0] - g[1]).median())

for k, df in {**cases, "Twitter links": tw_case}.items():
    print(f"{k:14s} within-author ratio = {within_author_ratio(df):.2f}")

The within-author comparison resolves A and D at once. It cannot separate B from C: in both, the same author's
category posts get fewer views. To tell a **penalty** from **lower appeal** we need to model **views and engagement
together**, with account-level effects, and estimate the penalty as a multiplicative factor on visibility
*net of* engagement.

This is what our model does *(Galeazzi et al., under review)*. Like you did today, we validate it on synthetic
platforms where the true penalty is known, then apply it to Twitter/X and YouTube.

<!-- INSTRUCTOR: add the model figure / equation here -->

**Open questions**
* If a platform shows penalised content to a less interested audience, the like rate drops too:
  can a penalty and low appeal ever be fully separated from the outside?
* Is a penalty that *emerges* from ranking by engagement a choice of the platform?
* Without data access (the Twitter/X academic API that provided view counts is gone), who audits the platforms?
  The EU **Digital Services Act** (art. 40) gives vetted researchers a right to data access.